# NB24 Stablecoins — Chapter 24

A chapter notebook for *Blockchain for Lawyers: A Reading Through Technology and Economics*. The code is the same as in the text.

- All code here is a **toy implementation** meant to make a principle visible. Do not use it with real assets, real private keys or real transactions.
- No external API is called, and everything runs without a network. Cells that need the bundled snapshot data carry that data inside the cell.
- Standard library only (run and checked on Python 3.13). There is no install cell.
- The outputs are the editor's own run, stored as produced. Running the cells again should give the same values (random seeds are fixed).

In [ ]:
# NB24-1: Diamond–Dybvig's two equilibria — with a demand deposit r1 > 1, for each total period-1 withdrawal fraction f, is the share from "waiting," V2, larger than "withdrawing," r1? (for study)
R, r1, t = 1.5, 1.2, 0.25          # period-2 output R, promised period-1 payment r1 (> 1), fraction t of type 1 (those who must consume at period 1)

def V2(f):                          # equation (7): the share of those who wait
    return max((1 - r1 * f) * R / (1 - f), 0.0) if f < 1 else 0.0

print(f"R = {R}, r1 = {r1}, type-1 fraction t = {t}. Withdrawal fraction at which the bank runs out, 1/r1 = {1 / r1:.3f}")
print(f"{'Period-1 f':>12} {'Waiting V2(f)':>16} {'Payout r1':>10} {'Type 2 best':>14}")
for f in [t, 0.40, 0.55, 0.70, 0.80, 0.83, 0.90, 1.00]:
    v = V2(f)
    print(f"{f:>12.2f} {v:>16.3f} {r1:>10.2f} {'wait' if v > r1 else 'withdraw (run)':>14}")
# The threshold where waiting becomes worse than withdrawing: (1 − r1 f)R/(1 − f) = r1  →  f* = (R − r1)/(r1 (R − 1))
f_star = (R - r1) / (r1 * (R - 1))
print(f"\nThreshold f* = (R − r1)/(r1(R − 1)) = {f_star:.3f}. If you believe others will withdraw more than this, withdrawing is your best response too: belief decides the outcome.")
print(f"At f = t = {t} (normal), type 2 waits and receives {V2(t):.3f} (> r1). At f = 1 (run), everyone shares an average of 1 and the back of the line gets 0: everyone is worse off than with no bank.")
print("\nAdd suspension of convertibility (period-1 payments stop when f reaches the suspension point) and the share of those who wait cannot fall below r1, so waiting becomes a dominant strategy for type 2 — when t is known.")

R = 1.5, r1 = 1.2, type-1 fraction t = 0.25. Withdrawal fraction at which the bank runs out, 1/r1 = 0.833
  Period-1 f    Waiting V2(f)  Payout r1    Type 2 best
        0.25            1.400       1.20           wait
        0.40            1.300       1.20           wait
        0.55            1.133       1.20 withdraw (run)
        0.70            0.800       1.20 withdraw (run)
        0.80            0.300       1.20 withdraw (run)
        0.83            0.035       1.20 withdraw (run)
        0.90            0.000       1.20 withdraw (run)
        1.00            0.000       1.20 withdraw (run)

Threshold f* = (R − r1)/(r1(R − 1)) = 0.500. If you believe others will withdraw more than this, withdrawing is your best response too: belief decides the outcome.
At f = t = 0.25 (normal), type 2 waits and receives 1.400 (> r1). At f = 1 (run), everyone shares an average of 1 and the back of the line gets 0: everyone is worse off than with no bank.

Add suspension of convertibility (pe

In [ ]:
# NB24-2: reserve composition × redemption demand → depeg feedback — types A, B and C are illustrations of structure, not the books of any coin (for study)
# Feedback rule (this book's simplification): if the redeemable ratio RR < 1, or the issuer closes the window, the secondary-market price moves down toward RR,
# and if the price is below 1, more holders come forward to redeem (by the 'sensitivity' below). Prices and quantities are all model variables.
def run_dynamics(reserves, q0, sensitivity=1.5, rounds=6, window_open=True):
    """reserves: [(name, weight, disposable share ℓ, fire-sale discount h)], weights sum to 1 (collateral ratio 1.0). q0: initial redemption demand fraction."""
    S = 1.0                                   # issuance normalized to 1
    q, price, history = q0, 1.0, []
    redeemed = 0.0
    for r in range(1, rounds + 1):
        liquid = sum(w * l * (1 - h) for _, w, l, h in reserves)        # value that can be disposed of right now (per issuance of 1)
        demand = min(q, 1.0) - redeemed
        rr = liquid / demand if demand > 0 else float("inf")
        if window_open and rr >= 1:
            redeemed += demand; price = 1.0
            history.append((r, q, round(rr, 2), price, "redeemed at par"))
            break
        price = max(min(rr, 1.0), 0.0) if window_open else max(price - 0.05, 0.0)   # if the window closes, the price slides (rule)
        history.append((r, round(q, 3), round(rr, 2), round(price, 3), "window open" if window_open else "window closed"))
        q = min(1.0, q + sensitivity * (1 - price) * (1 - q))          # if the price is below 1, more holders come forward to redeem
    return history

types = {
    "A Mostly cash and T-bills":            [("cash", 0.20, 1.0, 0.0), ("T-bills", 0.75, 1.0, 0.005), ("bank deposits", 0.05, 1.0, 0.0)],
    "B Deposits and CP mixed in":           [("cash", 0.05, 1.0, 0.0), ("T-bills", 0.40, 1.0, 0.005), ("bank deposits", 0.30, 1.0, 0.0), ("commercial paper", 0.25, 1.0, 0.05)],
    "B′ B with all bank deposits frozen":   [("cash", 0.05, 1.0, 0.0), ("T-bills", 0.40, 1.0, 0.005), ("bank deposits (frozen)", 0.30, 0.0, 0.0), ("commercial paper", 0.25, 1.0, 0.05)],
    "C Crypto-overcollateralized (initial collateral ratio 1.5 → 0.83 after a 45% sharp fall in collateral)": [("collateral crypto-asset — value left after a 45% sharp fall, fire-sale discount 15%", 1.5 * 0.55, 1.0, 0.15)],
}
for name, res in types.items():
    cr = sum(w for _, w, _, _ in res)
    print(f"\n[{name}] collateral ratio {cr:.2f}")
    for q0 in [0.10, 0.40, 0.80]:
        hist = run_dynamics(res, q0)
        last = hist[-1]
        print(f"  initial demand {q0:.0%}: {len(hist)} round(s) → demand {last[1]:.0%}, redeemable ratio {last[2]}, price {last[3]} ({last[4]})")

print("\n[B′ with the issuer closing the window] — with a secondary market, closing the window still carries the run into the price")
for row in run_dynamics(types["B′ B with all bank deposits frozen"], 0.40, window_open=False):
    print("  round {} demand {} RR {} price {} {}".format(*row))


[A Mostly cash and T-bills] collateral ratio 1.00
  initial demand 10%: 1 round(s) → demand 10%, redeemable ratio 9.96, price 1.0 (redeemed at par)
  initial demand 40%: 1 round(s) → demand 40%, redeemable ratio 2.49, price 1.0 (redeemed at par)
  initial demand 80%: 1 round(s) → demand 80%, redeemable ratio 1.25, price 1.0 (redeemed at par)

[B Deposits and CP mixed in] collateral ratio 1.00
  initial demand 10%: 1 round(s) → demand 10%, redeemable ratio 9.86, price 1.0 (redeemed at par)
  initial demand 40%: 1 round(s) → demand 40%, redeemable ratio 2.46, price 1.0 (redeemed at par)
  initial demand 80%: 1 round(s) → demand 80%, redeemable ratio 1.23, price 1.0 (redeemed at par)

[B′ B with all bank deposits frozen] collateral ratio 1.00
  initial demand 10%: 1 round(s) → demand 10%, redeemable ratio 6.85, price 1.0 (redeemed at par)
  initial demand 40%: 1 round(s) → demand 40%, redeemable ratio 1.71, price 1.0 (redeemed at par)
  initial demand 80%: 6 round(s) → demand 97%, redeem

In [ ]:
# NB24-3: reflexive feedback in the algorithmic type — does a spiral arise from just three things: the exchange promise "one S = 1 dollar's worth of L," the demand curve for L, and redemptions that grow once the backing breaks? (for study, model variables)
def algorithmic_spiral(S0, L0, p0, eps=1.0, rounds=10, base_redeem=0.05, panic=0.6, shock_round=3, shock=0.3):
    """Demand curve for L: p_L = D / L**(1+eps). With eps > 0, the price falls more than proportionally as supply rises (demand is inelastic; with eps=1, L market cap = D/L).
    Each round some S holders burn S and receive 1/p_L units of L. If the backing (L market cap / S) is below 1, the redemption fraction grows by panic.
    At shock_round the demand level D shrinks by a factor of shock. All numbers are model values."""
    S, L = float(S0), float(L0)
    D = p0 * L ** (1 + eps)
    print(f"{'Round':>5} {'S supply':>9} {'L supply':>11} {'p_L':>8} {'L mcap / S':>12} {'Redeem share':>12}")
    for r in range(1, rounds + 1):
        if r == shock_round:
            D *= shock                                        # demand shock (model variable)
        p_L = D / L ** (1 + eps)
        backing = L * p_L / S                                 # dollars' worth that all of L can pay back per S
        share = base_redeem + (panic * (1 - backing) if backing < 1 else 0.0)
        share = min(share, 0.9)
        print(f"{r:>5} {S:>9,.0f} {L:>11,.0f} {p_L:>8.4f} {backing:>12.2f} {share:>12.2f}")
        burn = S * share
        S -= burn
        L += burn / p_L                                       # mint new L worth "1 dollar" for each S burned
    return S, L, D / L ** (1 + eps)

S_end, L_end, p_end = algorithmic_spiral(S0=1_000, L0=2_000, p0=1.0)
print(f"\nEnd: S {S_end:,.0f}, L {L_end:,.0f}, p_L {p_end:.4f}.")
print("After the demand shock, once the market cap of L falls below S (once the BIS condition breaks), redemptions mint more L, the added supply pushes p_L down further, and that brings on more redemptions — a spiral.")
print("The numbers in this table are model values fitted to no data. What shows is only the shape: when the backing is \"demand for another token,\" the backing disappears when that demand disappears.")

Round  S supply    L supply      p_L   L mcap / S Redeem share
    1     1,000       2,000   1.0000         2.00         0.05
    2       950       2,050   0.9518         2.05         0.05
    3       902       2,100   0.2721         0.63         0.27
    4       659       2,996   0.1337         0.61         0.29
    5       471       4,400   0.0620         0.58         0.30
    6       328       6,699   0.0267         0.55         0.32
    7       222      10,663   0.0106         0.51         0.35
    8       145      17,966   0.0037         0.46         0.37
    9        91      32,602   0.0011         0.40         0.41
   10        54      65,409   0.0003         0.34         0.45

End: S 30, L 151,116, p_L 0.0001.
After the demand shock, once the market cap of L falls below S (once the BIS condition breaks), redemptions mint more L, the added supply pushes p_L down further, and that brings on more redemptions — a spiral.
The numbers in this table are model values fitted to no data.